In [3]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [5]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 1500  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'CMS_CTI'
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) '
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,threatAssess,associatedIndicators'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,confidence,threatAssessRating,threatAssessConfidence,...,lastObserved,associatedGroups.data,associatedIndicators.data,address,rating,sha256,description,md5,indicator,sources
0,14636698788988007,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-14T11:04:36Z,98,0.0,98.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152.59.177.138,CMS_CTI
1,14636698788988006,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-14T11:04:36Z,97,0.0,97.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,138.199.31.205,CMS_CTI
2,14636698788988005,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-14T11:04:36Z,98,0.0,98.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152.59.176.82,CMS_CTI
3,14636698788988004,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-14T11:04:36Z,92,0.5,51.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.225.28.228,CMS_CTI
4,14636698788988003,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Host,2026-05-14T11:04:36Z,91,0.0,91.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bj88-12.com,CMS_CTI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14742,5194324,2024-12-20T17:00:17Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2024-12-20T17:00:17Z,74,0.5,42.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,180.131.145.61,CMS_CTI
14743,5194311,2024-12-20T17:00:17Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2024-12-20T17:00:17Z,71,0.0,71.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.147.85.102,CMS_CTI
14744,5194303,2024-12-20T17:00:17Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2024-12-20T17:00:17Z,81,0.0,81.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,152.42.216.204,CMS_CTI
14745,5194286,2024-12-20T17:00:17Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Host,2024-12-20T17:00:17Z,89,0.0,89.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ssaresirect.pages.dev,CMS_CTI


In [10]:
# ──────────────────────────────────────────────────────────────────────────────
# Clean enrichment: TC enrich + direct VirusTotal ASN fallback for IPs
# ──────────────────────────────────────────────────────────────────────────────
import os
import urllib.parse
import requests
import pandas as pd
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

COL_PATH = "data.enrichment.data"   # adjust if API changes

# Load VirusTotal API key
load_dotenv(r"H:\HTOC\.env")
VT_API_KEY = os.getenv("VT_API_KEY")

if not VT_API_KEY:
    raise ValueError("VT_API_KEY not found in H:\\HTOC\\.env")

VT_HEADERS = {"x-apikey": VT_API_KEY}

# Determine candidate indicators
key_col = "indicator" if "indicator" in observed_src.columns else "summary"

# Provider allowlists
VT_TYPES = {"Address", "IPv4", "IPv6", "Host", "Domain", "URL", "File", "SHA1", "SHA256", "MD5"}
SHODAN_TYPES = {"Address", "IPv4", "IPv6"}
IP_TYPES = {"Address", "IPv4", "IPv6"}

# Build candidates and include ID when present
cols = [key_col, "type"]
if "id" in observed_src.columns:
    cols.append("id")

candidates = (
    observed_src[cols]
    .dropna(subset=[key_col])
    .astype({key_col: str})
    .drop_duplicates(subset=[key_col])
)

# Pre-filter supported enrichment types
candidates = candidates[
    candidates["type"].astype(str).str.strip().isin(VT_TYPES | SHODAN_TYPES)
].copy()

indicator_values = candidates[key_col].tolist()
display(f"Enriching {len(indicator_values)} indicators with TC enrich; direct VT ASN fallback for IPs...")

def _enrich_one(row_series):
    """Call ThreatConnect enrich API for one indicator."""
    value = row_series[key_col]
    typ = str(row_series.get("type", "") or "").strip()
    row_id = row_series.get("id")
    use_id = pd.notna(row_id) and str(row_id).strip().isdigit()

    try:
        # Prefer ID-based URL to avoid "More than one indicator matches" errors
        if use_id:
            indicator_id_or_value = str(int(float(row_id)))
        else:
            indicator_id_or_value = urllib.parse.quote(value, safe="")

        providers = []
        if typ in VT_TYPES:
            providers.append("VirusTotalV3")
        if typ in SHODAN_TYPES:
            providers.append("Shodan")

        if not providers:
            return (None, {key_col: value, "type": typ, "error": "No supported provider"})

        q = urllib.parse.urlencode({"type": providers}, doseq=True)
        enrich_url = f"/v3/indicators/{indicator_id_or_value}/enrich?{q}"

        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        resp = tc.api_request(ro)

        try:
            data = resp.json()
        except Exception:
            data = {
                "status": getattr(resp, "status_code", "n/a"),
                "raw": getattr(resp, "text", None),
            }

        data[key_col] = value
        return (data, None)

    except Exception as e:
        return (None, {key_col: value, "type": typ, "error": str(e)})

def fetch_vt_ip_asn(ip):
    """Direct VT lookup for IP ASN data missing from TC enrichment."""
    try:
        url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"
        r = requests.get(url, headers=VT_HEADERS, timeout=15)

        if r.status_code != 200:
            return {
                key_col: ip,
                "vt_asn": None,
                "vt_as_owner": None,
                "vt_network": None,
                "vt_rir": None,
                "vt_direct_status": r.status_code,
            }

        attrs = r.json().get("data", {}).get("attributes", {})

        return {
            key_col: ip,
            "vt_asn": attrs.get("asn"),
            "vt_as_owner": attrs.get("as_owner"),
            "vt_network": attrs.get("network"),
            "vt_rir": attrs.get("regional_internet_registry"),
            "vt_direct_status": r.status_code,
        }

    except Exception as e:
        return {
            key_col: ip,
            "vt_asn": None,
            "vt_as_owner": None,
            "vt_network": None,
            "vt_rir": None,
            "vt_direct_status": "error",
            "vt_direct_error": str(e),
        }

# ── ThreatConnect enrichment ───────────────────────────────────────────────
enriched_results = []
failed = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(_enrich_one, row): row for _, row in candidates.iterrows()}
    for future in as_completed(futures):
        result, err = future.result()
        if result is not None:
            enriched_results.append(result)
        else:
            failed.append(err)

# If nothing enriched, carry on with base
if not enriched_results:
    display("No ThreatConnect enrichment data retrieved.")
    recent_tags = observed_src.copy()
else:
    df_enriched = (
        pd.json_normalize(enriched_results)
        .drop_duplicates(subset=[key_col], keep="last")
    )

    # Merge enrichment block into base
    recent_tags = observed_src.merge(df_enriched, on=key_col, how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ─────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[[key_col, COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat[key_col] = exploded[key_col].values

        def _flatten_lists(values):
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            if all(not isinstance(v, (list, dict)) for v in flat):
                return list(pd.Series(flat).unique())
            return flat

        obj_cols = enrich_flat.select_dtypes("object").columns.difference([key_col])
        num_cols = enrich_flat.columns.difference(obj_cols.union({key_col}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
            .groupby(key_col, as_index=False)
            .agg(agg_dict)
        )

        recent_tags = (
            recent_tags
            .drop(columns=[COL_PATH], errors="ignore")
            .drop_duplicates(subset=[key_col])
            .merge(enrich_wide, on=key_col, how="left")
        )

    display(f"ThreatConnect enrichment complete for {recent_tags[key_col].notna().sum()} indicators.")

# ── Direct VirusTotal ASN fallback for IPs ─────────────────────────────────
ip_candidates = (
    candidates[candidates["type"].astype(str).str.strip().isin(IP_TYPES)][key_col]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

vt_results = []

if ip_candidates:
    display(f"Running direct VT ASN lookup for {len(ip_candidates)} IP indicators...")

    with ThreadPoolExecutor(max_workers=6) as executor:
        futures = {executor.submit(fetch_vt_ip_asn, ip): ip for ip in ip_candidates}
        for future in as_completed(futures):
            res = future.result()
            if res:
                vt_results.append(res)

if vt_results:
    df_vt = pd.DataFrame(vt_results).drop_duplicates(subset=[key_col], keep="last")
    recent_tags = recent_tags.merge(df_vt, on=key_col, how="left")
    display(f"Direct VT ASN enrichment added for {df_vt['vt_asn'].notna().sum()} IPs.")
else:
    display("No direct VT ASN enrichment data retrieved.")

# ── Failure summary ────────────────────────────────────────────────────────
if failed:
    fail_df = pd.DataFrame(failed)
    display(f"{len(failed)} indicators failed TC enrichment, showing up to 10:")
    display(fail_df.head(10))

# ── Preview key enrichment columns ─────────────────────────────────────────
preview_cols = [
    c for c in [
        key_col,
        "type",
        "vt_asn",
        "vt_as_owner",
        "vt_network",
        "vt_rir",
        "vt_direct_status",
        "enrich_hostNames",
        "enrich_domains",
        "enrich_tags",
        "enrich_vtMaliciousCount",
    ]
    if c in recent_tags.columns
]

display(recent_tags[preview_cols])

'Enriching 14733 indicators with TC enrich; direct VT ASN fallback for IPs...'

Status Code: 404
Failed API Response: b'{"errCode":"0x1004","message":"Indicator could not be found.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The URL https://static.alfreshup.com/Xgd37BKqtSdK7BK3i0z3E0djiczMucfjHgbB1ol3tSdD7BKqkw== cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The URL https://static.alfreshup.com/Xgd37BKqtSdK7BK3i0z3E0djiczMucfjHgbB1ol3tSdD7BKqkw== cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The File 7FBCD9249BE84CF53B67220A0EA97146 : D5D63A4DC883D2A753E41B965B86A3DDBD8F908A : F673C918FDC91EC19A944B89078F50CAF4B15E20023FEAFB500C1C4F533028BD cannot be enriched with Shodan because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errC

'ThreatConnect enrichment complete for 14747 indicators.'

'Running direct VT ASN lookup for 10397 IP indicators...'

'Direct VT ASN enrichment added for 10024 IPs.'

'10929 indicators failed TC enrichment, showing up to 10:'

,indicator,type,error
0,152.59.177.138,Address,"b'{""errCode"":""0x1004"",""message"":""Indicator cou..."
1,185.225.28.236,Address,"b'{""errCode"":""0x1001"",""message"":""The URL https..."
2,41.123.242.161,Address,"b'{""errCode"":""0x1001"",""message"":""The URL https..."
3,31.171.155.142,Address,"b'{""errCode"":""0x1001"",""message"":""The File 7FBC..."
4,194.146.92.224,Address,"b'{""errCode"":""0x1001"",""message"":""The File C3FD..."
5,168.144.37.240,Address,"b'{""errCode"":""0x1001"",""message"":""The File FF2A..."
6,195.178.110.135,Address,"b'{""errCode"":""0x1001"",""message"":""The URL https..."
7,152.59.178.236,Address,"b'{""errCode"":""0x1001"",""message"":""The File 7E60..."
8,195.63.14.210,Address,"b'{""errCode"":""0x1001"",""message"":""The URL tcp:/..."
9,103.168.66.178,Address,"b'{""errCode"":""0x1001"",""message"":""The Host prec..."


,indicator,type,vt_asn,vt_as_owner,vt_network,vt_rir,vt_direct_status,enrich_hostNames,enrich_domains,enrich_tags,enrich_vtMaliciousCount
0,152.59.177.138,Address,55836.0,Reliance Jio Infocomm Limited,152.56.0.0/14,APNIC,200.0,NaN,NaN,NaN,NaN
1,138.199.31.205,Address,212238.0,Datacamp Limited,138.199.28.0/22,RIPE NCC,200.0,[unn-138-199-31-205.datapacket.com],[datapacket.com],None,1.0
2,152.59.176.82,Address,55836.0,Reliance Jio Infocomm Limited,152.56.0.0/14,APNIC,200.0,None,None,[eol-product],0.0
3,185.225.28.228,Address,205119.0,TELEKS DOOEL Skopje,185.225.28.0/22,RIPE NCC,200.0,None,None,[eol-product],0.0
4,bj88-12.com,Host,NaN,NaN,NaN,NaN,NaN,None,None,None,0.0
...,...,...,...,...,...,...,...,...,...,...,...
14742,180.131.145.61,Address,16276.0,OVH SAS,180.131.145.0/24,ARIN,200.0,NaN,NaN,NaN,NaN
14743,2.147.85.102,Address,44244.0,Iran Cell Service and Communication Company,2.144.0.0/14,RIPE NCC,200.0,NaN,NaN,NaN,NaN
14744,152.42.216.204,Address,14061.0,"DigitalOcean, LLC",152.42.128.0/17,APNIC,200.0,NaN,NaN,NaN,NaN
14745,ssaresirect.pages.dev,Host,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
recent_tags[recent_tags['indicator'] == '152.59.177.138']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,confidence,threatAssessRating,threatAssessConfidence,...,enrich_org,enrich_tags,enrich_type,enrich_vulnerabilities,enrich_vtMaliciousCount,vt_asn,vt_as_owner,vt_network,vt_rir,vt_direct_status
0,14636698788988007,2026-05-14T11:00:07Z,40,CMS_CTI,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-14T11:04:36Z,98,0.0,98.0,...,NaN,NaN,NaN,NaN,NaN,55836.0,Reliance Jio Infocomm Limited,152.56.0.0/14,APNIC,200.0


import requests

VT_API_KEY = "34e957761e642821555bd3078c90ca035674741d0a7d9da1ee1ea31990d5b4a9"
ip = "152.59.177.138"
url = f"https://www.virustotal.com/api/v3/ip_addresses/{ip}"

headers = {"x-apikey": VT_API_KEY}

r = requests.get(url, headers=headers)
vt = r.json()

asn = vt.get("data", {}).get("attributes", {}).get("asn")
as_owner = vt.get("data", {}).get("attributes", {}).get("as_owner")
network = vt.get("data", {}).get("attributes", {}).get("network")

print(asn, as_owner, network)

In [16]:
import math


def _vt_asn_display(x):
    """Show ASN without a spurious `.0` (e.g. 10029.0 -> '10029')."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, bool):
        return str(x)
    if isinstance(x, int):
        return str(x)
    if isinstance(x, float):
        if math.isfinite(x) and x == int(x):
            return str(int(x))
        return str(x)
    s = str(x).strip()
    if s.lower() in ('nan', 'none', ''):
        return None
    try:
        f = float(s)
        if math.isfinite(f) and f == int(f):
            return str(int(f))
    except ValueError:
        pass
    return s


if 'enrich_asn' in recent_tags.columns and 'vt_asn' not in recent_tags.columns:
    recent_tags['vt_asn'] = recent_tags['enrich_asn']

if 'vt_asn' in recent_tags.columns:
    recent_tags['vt_asn'] = recent_tags['vt_asn'].map(_vt_asn_display).astype('object')
    _asn_for_distinct = recent_tags['vt_asn'].dropna()
    distinct_vt_asn = sorted(_asn_for_distinct.unique())
    display(f'distinct vt_asn ({len(distinct_vt_asn)} values)')
    display(distinct_vt_asn)

    WATCHLIST_ASNS = frozenset(
        {
            '398324',
            '398705',
            '398722',
            '213412',
            '7806',
            '61244',
            '205113',
            '20940',
            '16625',
            '18680',
            '13335',
            '16509',
            '54113',
            '15169',
            '8075',
            '3356',
            '3491',
            '15133',
            '22822',
        }
    )
    _present = set(distinct_vt_asn)
    watch_hits = sorted(_present & WATCHLIST_ASNS)
    display(
        f"Watchlist check: {len(watch_hits)} of {len(WATCHLIST_ASNS)} watchlist ASNs appear in vt_asn."
    )
    display(watch_hits if watch_hits else 'No watchlist ASNs in this batch.')
    if watch_hits:
        _hit_mask = recent_tags['vt_asn'].isin(watch_hits)
        _cols = [c for c in [key_col, 'vt_asn', 'type'] if c in recent_tags.columns]
        display(recent_tags.loc[_hit_mask, _cols].drop_duplicates())

preview_cols = [
    c
    for c in [
        key_col,
        'enrich_hostNames',
        'enrich_domains',
        'enrich_tags',
        'enrich_vtMaliciousCount',
        'vt_asn',
    ]
    if c in recent_tags.columns
]

'distinct vt_asn (1119 values)'

['10029',
 '10139',
 '10429',
 '10620',
 '10796',
 '11320',
 '11351',
 '1136',
 '11377',
 '11404',
 '11426',
 '11427',
 '11664',
 '11686',
 '11878',
 '12091',
 '12252',
 '12322',
 '12389',
 '12479',
 '12616',
 '12735',
 '12876',
 '12975',
 '12978',
 '12989',
 '13022',
 '13030',
 '131178',
 '131199',
 '131207',
 '131275',
 '131366',
 '131386',
 '131429',
 '131483',
 '131965',
 '132116',
 '13213',
 '132165',
 '132199',
 '132203',
 '132296',
 '132298',
 '132323',
 '13237',
 '132372',
 '132445',
 '132585',
 '132768',
 '132817',
 '132825',
 '132883',
 '133001',
 '133037',
 '133238',
 '133275',
 '13335',
 '133398',
 '133524',
 '133605',
 '133623',
 '133648',
 '133652',
 '133661',
 '133693',
 '133706',
 '133774',
 '133776',
 '133944',
 '133982',
 '134022',
 '134294',
 '134319',
 '134375',
 '134599',
 '134601',
 '134707',
 '134762',
 '134768',
 '134771',
 '134773',
 '134806',
 '134810',
 '13489',
 '134928',
 '134943',
 '134970',
 '135139',
 '135239',
 '135247',
 '135300',
 '135310',
 '13536',


'Watchlist check: 6 of 19 watchlist ASNs appear in vt_asn.'

['13335', '15169', '16509', '398722', '54113', '8075']

,indicator,vt_asn,type
34,199.45.154.113,398722,Address
123,18.223.24.218,16509,Address
1773,199.45.154.177,398722,Address
1834,199.45.154.189,398722,Address
2006,199.45.155.111,398722,Address
...,...,...,...
10815,45.72.112.245,16509,Address
11085,144.168.254.66,16509,Address
11302,45.146.128.129,16509,Address
11692,45.72.85.172,16509,Address
